# 03 — Customer Analytics
**Build customer-level behavioral summaries from cleaned purchase data.**


## Objective
- Define the purpose of this stage.
- Record key decisions and assumptions.
- Keep outputs reproducible and easy to trace to the previous stage.


## Planned analysis
- Customer purchase activity
- Purchase sessions
- Monetary behavior
- Average transaction value
- Basket-size proxy based on purchase events/session
- Customer activity distributions
- Customer-level summary dataset for RFM


In [ ]:
# Load cleaned purchase data from Parquet.
# Build customer-level aggregates with PySpark.


## Output
Save the reusable customer summary as Parquet for the RFM stage.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ECommerceCustomerBehaviorAnalytics-03-CustomerAnalytics")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/28 21:11:22 WARN Utils: Your hostname, Vaishnavis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.102 instead (on interface en0)
26/09/28 21:11:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/vaishnavipoojary/Desktop/ecommerce-customer-behavior-analytics/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/28 21:11:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

Spark version: 4.2.0


In [2]:
CLEANED_PATH = "../data/cleaned_ecommerce_oct.parquet"

df_clean = spark.read.parquet(CLEANED_PATH)

print("Rows:", df_clean.count())
print("Columns:", len(df_clean.columns))

df_clean.printSchema()

Rows: 42380089
Columns: 9
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [3]:
from pyspark.sql.functions import col

purchase_df = (
    df_clean
    .filter(col("event_type") == "purchase")
)

print("Purchase events:", purchase_df.count())

Purchase events: 742849


In [4]:
from pyspark.sql.functions import (
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    min,
    max
)

customer_summary = (
    purchase_df
    .groupBy("user_id")
    .agg(
        count("*").alias("purchase_events"),
        countDistinct("user_session").alias("purchase_sessions"),
        countDistinct("product_id").alias("unique_products"),
        spark_sum("price").alias("total_spend"),
        avg("price").alias("avg_purchase_value"),
        min("event_time").alias("first_purchase"),
        max("event_time").alias("last_purchase")
    )
)

print("Customers with purchases:", customer_summary.count())
customer_summary.show(10, truncate=False)

Customers with purchases: 347118


[Stage 13:========================================>               (10 + 4) / 14]

+---------+---------------+-----------------+---------------+-----------------+------------------+-------------------+-------------------+
|user_id  |purchase_events|purchase_sessions|unique_products|total_spend      |avg_purchase_value|first_purchase     |last_purchase      |
+---------+---------------+-----------------+---------------+-----------------+------------------+-------------------+-------------------+
|548082482|1              |1                |1              |182.76           |182.76            |2019-10-13 17:00:44|2019-10-13 17:00:44|
|514338009|31             |26               |12             |2686.73          |86.66870967741936 |2019-10-09 21:38:42|2019-10-30 23:05:57|
|546491844|2              |1                |1              |50.96            |25.48             |2019-10-11 18:59:08|2019-10-11 19:05:29|
|533816329|5              |3                |3              |441.35           |88.27000000000001 |2019-10-13 13:07:31|2019-10-14 13:58:42|
|517869931|3              |

In [5]:
customer_summary.select(
    "purchase_events",
    "purchase_sessions",
    "unique_products",
    "total_spend",
    "avg_purchase_value"
).describe().show()

26/09/28 21:05:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 21:=====>                                                  (1 + 10) / 11]

+-------+------------------+-----------------+------------------+-----------------+------------------+
|summary|   purchase_events|purchase_sessions|   unique_products|      total_spend|avg_purchase_value|
+-------+------------------+-----------------+------------------+-----------------+------------------+
|  count|            347118|           347118|            347118|           347118|            347118|
|   mean|2.1400474766505915|1.813677193346355| 1.608916852482441|662.4764554704822| 278.0301792343678|
| stddev| 3.639296123909378|2.437539163529341|1.5439691833503346|2074.370474528532| 311.2153189231909|
|    min|                 1|                1|                 1|             0.88|              0.88|
|    max|               322|              152|                65|265569.5200000001|           2574.04|
+-------+------------------+-----------------+------------------+-----------------+------------------+



In [5]:
from pyspark.sql.functions import when

purchase_frequency_buckets = (
    customer_summary
    .withColumn(
        "frequency_bucket",
        when(col("purchase_events") == 1, "1 purchase")
        .when(col("purchase_events") == 2, "2 purchases")
        .when(col("purchase_events").between(3, 5), "3–5 purchases")
        .when(col("purchase_events").between(6, 10), "6–10 purchases")
        .otherwise("11+ purchases")
    )
    .groupBy("frequency_bucket")
    .count()
)

purchase_frequency_buckets.show()

[Stage 19:========================================>               (10 + 4) / 14]

+----------------+------+
|frequency_bucket| count|
+----------------+------+
|      1 purchase|215691|
|   11+ purchases|  6607|
|  6–10 purchases| 12938|
|     2 purchases| 65574|
|   3–5 purchases| 46308|
+----------------+------+



In [6]:
from pyspark.sql.functions import sum as spark_sum, when, col

purchasing_users = customer_summary.select("user_id")

customer_activity = (
    df_clean
    .join(purchasing_users, on="user_id", how="left_semi")
    .groupBy("user_id")
    .agg(
        spark_sum(when(col("event_type") == "view", 1).otherwise(0))
            .alias("view_events"),
        spark_sum(when(col("event_type") == "cart", 1).otherwise(0))
            .alias("cart_events"),
        spark_sum(when(col("event_type") == "purchase", 1).otherwise(0))
            .alias("purchase_events")
    )
)

print("Customers with activity:", customer_activity.count())
customer_activity.show(10, truncate=False)

Customers with activity: 347118


+---------+-----------+-----------+---------------+
|user_id  |view_events|cart_events|purchase_events|
+---------+-----------+-----------+---------------+
|512517959|16         |0          |1              |
|545035858|40         |5          |2              |
|531913156|91         |4          |1              |
|535568683|472        |25         |9              |
|514292886|4          |0          |1              |
|516054250|8          |0          |3              |
|539885835|105        |5          |3              |
|518667619|7          |1          |1              |
|549847476|88         |3          |2              |
|513512236|40         |0          |1              |
+---------+-----------+-----------+---------------+
only showing top 10 rows


In [7]:
customer_analytics = (
    customer_summary
    .join(customer_activity, on="user_id", how="left")
)

print("Customers:", customer_analytics.count())
print("Columns:", len(customer_analytics.columns))

customer_analytics.printSchema()
customer_analytics.show(10, truncate=False)

Customers: 347118
Columns: 11
root
 |-- user_id: integer (nullable = true)
 |-- purchase_events: long (nullable = false)
 |-- purchase_sessions: long (nullable = false)
 |-- unique_products: long (nullable = false)
 |-- total_spend: double (nullable = true)
 |-- avg_purchase_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- view_events: long (nullable = true)
 |-- cart_events: long (nullable = true)
 |-- purchase_events: long (nullable = true)



+---------+---------------+-----------------+---------------+-----------------+------------------+-------------------+-------------------+-----------+-----------+---------------+
|user_id  |purchase_events|purchase_sessions|unique_products|total_spend      |avg_purchase_value|first_purchase     |last_purchase      |view_events|cart_events|purchase_events|
+---------+---------------+-----------------+---------------+-----------------+------------------+-------------------+-------------------+-----------+-----------+---------------+
|514338009|31             |26               |12             |2686.73          |86.66870967741936 |2019-10-09 21:38:42|2019-10-30 23:05:57|226        |2          |31             |
|517869931|3              |2                |2              |810.6400000000001|270.21333333333337|2019-10-13 15:36:54|2019-10-25 19:53:31|9          |2          |3              |
|521351169|2              |1                |1              |56.62            |28.31             |2019-10

In [8]:
customer_analytics = (
    customer_summary
    .join(
        customer_activity.select(
            "user_id",
            "view_events",
            "cart_events"
        ),
        on="user_id",
        how="left"
    )
)

print("Customers:", customer_analytics.count())
print("Columns:", len(customer_analytics.columns))

customer_analytics.printSchema()

Customers: 347118
Columns: 10
root
 |-- user_id: integer (nullable = true)
 |-- purchase_events: long (nullable = false)
 |-- purchase_sessions: long (nullable = false)
 |-- unique_products: long (nullable = false)
 |-- total_spend: double (nullable = true)
 |-- avg_purchase_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- view_events: long (nullable = true)
 |-- cart_events: long (nullable = true)



In [9]:
from pyspark.sql.functions import round

customer_analytics = (
    customer_analytics
    .withColumn(
        "view_to_cart_rate",
        when(
            col("view_events") > 0,
            col("cart_events") / col("view_events")
        ).otherwise(0)
    )
    .withColumn(
        "cart_to_purchase_rate",
        when(
            col("cart_events") > 0,
            col("purchase_events") / col("cart_events")
        ).otherwise(0)
    )
    .withColumn(
        "purchase_to_view_rate",
        when(
            col("view_events") > 0,
            col("purchase_events") / col("view_events")
        ).otherwise(0)
    )
)

customer_analytics.select(
    "user_id",
    "view_events",
    "cart_events",
    "purchase_events",
    "view_to_cart_rate",
    "cart_to_purchase_rate",
    "purchase_to_view_rate"
).show(10, truncate=False)

+---------+-----------+-----------+---------------+--------------------+---------------------+---------------------+
|user_id  |view_events|cart_events|purchase_events|view_to_cart_rate   |cart_to_purchase_rate|purchase_to_view_rate|
+---------+-----------+-----------+---------------+--------------------+---------------------+---------------------+
|512517959|16         |0          |1              |0.0                 |0.0                  |0.0625               |
|514292886|4          |0          |1              |0.0                 |0.0                  |0.25                 |
|516054250|8          |0          |3              |0.0                 |0.0                  |0.375                |
|516315087|145        |1          |8              |0.006896551724137931|8.0                  |0.05517241379310345  |
|518667619|7          |1          |1              |0.14285714285714285 |1.0                  |0.14285714285714285  |
|539885835|105        |5          |3              |0.04761904761

In [10]:
customer_analytics = (
    customer_analytics
    .withColumn(
        "spend_per_purchase_session",
        when(
            col("purchase_sessions") > 0,
            col("total_spend") / col("purchase_sessions")
        ).otherwise(0)
    )
    .withColumn(
        "purchases_per_session",
        when(
            col("purchase_sessions") > 0,
            col("purchase_events") / col("purchase_sessions")
        ).otherwise(0)
    )
    .withColumn(
        "views_per_purchase",
        when(
            col("purchase_events") > 0,
            col("view_events") / col("purchase_events")
        ).otherwise(0)
    )
)

customer_analytics.select(
    "user_id",
    "total_spend",
    "purchase_sessions",
    "purchase_events",
    "view_events",
    "spend_per_purchase_session",
    "purchases_per_session",
    "views_per_purchase"
).show(10, truncate=False)

+---------+------------------+-----------------+---------------+-----------+--------------------------+---------------------+------------------+
|user_id  |total_spend       |purchase_sessions|purchase_events|view_events|spend_per_purchase_session|purchases_per_session|views_per_purchase|
+---------+------------------+-----------------+---------------+-----------+--------------------------+---------------------+------------------+
|512440362|7706.400000000001 |8                |10             |180        |963.3000000000001         |1.25                 |18.0              |
|513271955|143.38            |3                |3              |85         |47.79333333333333         |1.0                  |28.333333333333332|
|514338009|2686.7300000000005|26               |31             |226        |103.33576923076924        |1.1923076923076923   |7.290322580645161 |
|521674679|4276.46           |12               |16             |75         |356.37166666666667        |1.3333333333333333   |4.687

In [11]:
customer_analytics.select(
    "purchase_events",
    "purchase_sessions",
    "unique_products",
    "total_spend",
    "avg_purchase_value",
    "view_events",
    "cart_events",
    "view_to_cart_rate",
    "cart_to_purchase_rate",
    "purchase_to_view_rate",
    "spend_per_purchase_session",
    "purchases_per_session",
    "views_per_purchase"
).describe().show()

26/09/28 21:17:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 92:>                                                       (0 + 10) / 11]

+-------+------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-------------------+---------------------+---------------------+--------------------------+---------------------+------------------+
|summary|   purchase_events|purchase_sessions|   unique_products|       total_spend|avg_purchase_value|       view_events|      cart_events|  view_to_cart_rate|cart_to_purchase_rate|purchase_to_view_rate|spend_per_purchase_session|purchases_per_session|views_per_purchase|
+-------+------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-------------------+---------------------+---------------------+--------------------------+---------------------+------------------+
|  count|            347118|           347118|            347118|            347118|            347118|            347118|           347118|             347118|               347118

In [12]:
customer_analytics.select(
    "total_spend"
).summary(
    "count",
    "25%",
    "50%",
    "75%",
    "90%",
    "95%",
    "99%",
    "max"
).show()

[Stage 98:============================================>           (11 + 3) / 14]

+-------+-----------------+
|summary|      total_spend|
+-------+-----------------+
|  count|           347118|
|    25%|           107.58|
|    50%|           246.52|
|    75%|           594.55|
|    90%|          1418.05|
|    95%|          2336.84|
|    99%|6634.219999999999|
|    max|265569.5200000001|
+-------+-----------------+



In [13]:
customer_analytics.select(
    "avg_purchase_value"
).summary(
    "count",
    "25%",
    "50%",
    "75%",
    "90%",
    "95%",
    "99%",
    "max"
).show()

+-------+------------------+
|summary|avg_purchase_value|
+-------+------------------+
|  count|            347118|
|    25%| 82.37142857142855|
|    50%|             172.0|
|    75%|             331.8|
|    90%| 703.6033333333334|
|    95%|            986.69|
|    99%|           1521.27|
|    max|           2574.04|
+-------+------------------+



In [14]:
CUSTOMER_ANALYTICS_PATH = "../data/customer_analytics_oct.parquet"

customer_analytics.write.mode("overwrite").parquet(
    CUSTOMER_ANALYTICS_PATH
)

print("Customer analytics saved successfully.")
print("Path:", CUSTOMER_ANALYTICS_PATH)

26/09/28 21:18:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/28 21:18:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/28 21:18:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers


Customer analytics saved successfully.
Path: ../data/customer_analytics_oct.parquet


26/09/28 21:18:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/28 21:18:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [15]:
from pyspark.sql.functions import when, col

customer_analytics = (
    customer_analytics
    .withColumn(
        "spend_segment",
        when(col("total_spend") < 100, "Low Spend")
        .when(col("total_spend") < 500, "Medium Spend")
        .when(col("total_spend") < 2000, "High Spend")
        .otherwise("Very High Spend")
    )
)

customer_analytics.groupBy("spend_segment").count().show()

+---------------+------+
|  spend_segment| count|
+---------------+------+
|Very High Spend| 21557|
|      Low Spend| 82096|
|   Medium Spend|163561|
|     High Spend| 79904|
+---------------+------+



In [16]:
from pyspark.sql.functions import max as spark_max, datediff, to_date

analysis_date = (
    customer_analytics
    .select(spark_max("last_purchase").alias("max_purchase_date"))
    .collect()[0]["max_purchase_date"]
)

print("Analysis reference date:", analysis_date)

Analysis reference date: 2019-11-01 05:29:16


In [17]:
from pyspark.sql.functions import datediff, to_date, lit

customer_analytics = (
    customer_analytics
    .withColumn(
        "recency_days",
        datediff(
            to_date(lit(analysis_date)),
            to_date(col("last_purchase"))
        )
    )
)

customer_analytics.select(
    "user_id",
    "last_purchase",
    "recency_days"
).show(10, truncate=False)

+---------+-------------------+------------+
|user_id  |last_purchase      |recency_days|
+---------+-------------------+------------+
|514292886|2019-10-13 12:07:40|19          |
|516054250|2019-10-19 12:03:34|13          |
|539885835|2019-10-29 10:11:22|3           |
|554469441|2019-10-13 12:54:50|19          |
|551332368|2019-10-13 12:15:07|19          |
|512517959|2019-10-13 12:21:40|19          |
|547254938|2019-10-29 17:04:04|3           |
|558462470|2019-10-13 13:58:07|19          |
|557014082|2019-10-13 12:35:08|19          |
|518667619|2019-10-13 12:35:55|19          |
+---------+-------------------+------------+
only showing top 10 rows


In [18]:
rfm_data = (
    customer_analytics
    .select(
        "user_id",
        "recency_days",
        "purchase_events",
        "total_spend"
    )
    .withColumnRenamed("purchase_events", "frequency")
    .withColumnRenamed("total_spend", "monetary")
)

print("RFM customers:", rfm_data.count())

rfm_data.show(10, truncate=False)

RFM customers: 347118
+---------+------------+---------+------------------+
|user_id  |recency_days|frequency|monetary          |
+---------+------------+---------+------------------+
|514292886|19          |1        |28.42             |
|516054250|13          |3        |334.14            |
|539885835|3           |3        |534.6600000000001 |
|554469441|19          |2        |462.82            |
|551332368|19          |2        |453.66999999999996|
|512517959|19          |1        |107.5             |
|547254938|3           |25       |6008.74           |
|558462470|19          |4        |1127.69           |
|557014082|19          |1        |69.59             |
|518667619|19          |1        |341.66            |
+---------+------------+---------+------------------+
only showing top 10 rows


In [19]:
FINAL_CUSTOMER_ANALYTICS_PATH = "../data/customer_analytics_oct.parquet"

customer_analytics.write.mode("overwrite").parquet(
    FINAL_CUSTOMER_ANALYTICS_PATH
)

print("Final customer analytics saved successfully.")
print("Rows:", customer_analytics.count())
print("Columns:", len(customer_analytics.columns))
print("Path:", FINAL_CUSTOMER_ANALYTICS_PATH)

26/09/28 21:23:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/28 21:23:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/28 21:23:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/28 21:23:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/28 21:23:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Final customer analytics saved successfully.
Rows: 347118
Columns: 18
Path: ../data/customer_analytics_oct.parquet


In [21]:
print("========== NOTEBOOK 03 FINAL VALIDATION ==========")

print("Customers:", customer_analytics.count())
print("Columns:", len(customer_analytics.columns))

print("\nColumns:")
for c in customer_analytics.columns:
    print("-", c)

print("\nNull counts:")
customer_analytics.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in customer_analytics.columns
]).show(truncate=False)

========== NOTEBOOK 03 FINAL VALIDATION ==========
Customers: 347118
Columns: 18

Columns:
- user_id
- purchase_events
- purchase_sessions
- unique_products
- total_spend
- avg_purchase_value
- first_purchase
- last_purchase
- view_events
- cart_events
- view_to_cart_rate
- cart_to_purchase_rate
- purchase_to_view_rate
- spend_per_purchase_session
- purchases_per_session
- views_per_purchase
- spend_segment
- recency_days

Null counts:


+-------+---------------+-----------------+---------------+-----------+------------------+--------------+-------------+-----------+-----------+-----------------+---------------------+---------------------+--------------------------+---------------------+------------------+-------------+------------+
|user_id|purchase_events|purchase_sessions|unique_products|total_spend|avg_purchase_value|first_purchase|last_purchase|view_events|cart_events|view_to_cart_rate|cart_to_purchase_rate|purchase_to_view_rate|spend_per_purchase_session|purchases_per_session|views_per_purchase|spend_segment|recency_days|
+-------+---------------+-----------------+---------------+-----------+------------------+--------------+-------------+-----------+-----------+-----------------+---------------------+---------------------+--------------------------+---------------------+------------------+-------------+------------+
|0      |0              |0                |0              |0          |0                 |0      

## Final Summary

This notebook performs customer-level analytics on the cleaned October 2019 e-commerce dataset.

### Key Outcomes

- Identified **347,118 customers with at least one purchase event**.
- Created customer-level purchase metrics including:
  - Purchase frequency
  - Purchase sessions
  - Unique products purchased
  - Total spend
  - Average purchase value
  - First and last purchase timestamps
- Created customer engagement metrics based on:
  - View events
  - Cart events
  - Purchase events
  - View-to-cart event ratio
  - Cart-to-purchase event ratio
  - Purchase-to-view event ratio
- Created behavioral spending metrics:
  - Spend per purchase session
  - Purchases per purchase session
  - Views per purchase
- Examined customer spending distributions and identified strong right-skew in total customer spending and average purchase value.
- Created descriptive spending segments:
  - Low Spend
  - Medium Spend
  - High Spend
  - Very High Spend
- Calculated customer recency using the latest purchase timestamp in the October dataset as the analysis reference point.
- Prepared the core RFM attributes:
  - **Recency:** `recency_days`
  - **Frequency:** `purchase_events`
  - **Monetary:** `total_spend`

### Final Customer Analytics Dataset

The final customer-level dataset contains **347,118 customers and 18 features**, with **no missing values** across the generated customer analytics fields.

The dataset has been saved locally as:

`../data/customer_analytics_oct.parquet`

This customer-level dataset will serve as the input for the **Enhanced RFM analysis and customer segmentation** in the subsequent notebooks.

### Important Data Consideration

The source dataset does not contain an explicit order or transaction identifier. Therefore, purchase frequency and purchase-session metrics are based on the available **purchase events and `user_session` values**, respectively, rather than formal order-level transactions.